In [16]:
import pandas as pd
import numpy as np
from collections import defaultdict

def load_letor_file(path):
    """
    Парсер файла в формате LETOR
    """
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            relevance = int(parts[0])
            qid = int(parts[1].split(':')[1])

            features = {}
            for item in parts[2:]:
                if item.startswith('#'):
                    break
                fid, val = item.split(':')
                features[int(fid)] = float(val)

            rows.append({'relevance': relevance, 'qid': qid, **features})

    df = pd.DataFrame(rows)
    return df

# Загружаем Fold1
train = load_letor_file('/Users/ivan/Desktop/learning-to-rank/data/train.txt')
vali  = load_letor_file('/Users/ivan/Desktop/learning-to-rank/data/vali.txt')
test  = load_letor_file('/Users/ivan/Desktop/learning-to-rank/data/test.txt')

print("Train shape:", train.shape)
print("Vali shape:", vali.shape)
print("Test shape:", test.shape)

print("\nУникальных запросов (qid) в train:", train['qid'].nunique())
print("Распределение relevance:")
print(train['relevance'].value_counts().sort_index())

Train shape: (9630, 48)
Vali shape: (2707, 48)
Test shape: (2874, 48)

Уникальных запросов (qid) в train: 471
Распределение relevance:
relevance
0    7820
1    1223
2     587
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score
import lightgbm as lgb

# Признаки
feature_cols = [c for c in train.columns if c not in ['relevance', 'qid']]

X_train = train[feature_cols]
y_train = train['relevance']
qid_train = train['qid']

X_vali = vali[feature_cols]
y_vali = vali['relevance']
qid_vali = vali['qid']

X_test = test[feature_cols]
y_test = test['relevance']
qid_test = test['qid']

print("Количество признаков:", len(feature_cols))

Количество признаков: 46


In [9]:
# Pointwise: обычный регрессор
model_pointwise = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_pointwise.fit(
    X_train, y_train,
    eval_set=[(X_vali, y_vali)],
    eval_metric='l2',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# Предсказания
vali_pred = model_pointwise.predict(X_vali)
test_pred = model_pointwise.predict(X_test)

/Users/ivan/Desktop/learning-to-rank/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9233
[LightGBM] [Info] Number of data points in the train set: 9630, number of used features: 40
[LightGBM] [Info] Start training from score 0.248910


In [10]:
def calculate_ndcg(y_true, y_pred, qids, k=10):
    """
    Считает средний NDCG@k по запросам
    """
    df = pd.DataFrame({
        'qid': qids,
        'y_true': y_true,
        'y_pred': y_pred
    })

    ndcg_scores = []
    for qid, group in df.groupby('qid'):
        if len(group) < 2:
            continue
        true_relevance = group['y_true'].values.reshape(1, -1)
        pred_scores = group['y_pred'].values.reshape(1, -1)
        score = ndcg_score(true_relevance, pred_scores, k=k)
        ndcg_scores.append(score)

    return np.mean(ndcg_scores)

ndcg_vali = calculate_ndcg(y_vali, vali_pred, qid_vali, k=10)
ndcg_test = calculate_ndcg(y_test, test_pred, qid_test, k=10)

print(f"Pointwise NDCG@10 на валидации: {ndcg_vali:.4f}")
print(f"Pointwise NDCG@10 на тесте: {ndcg_test:.4f}")


Pointwise NDCG@10 на валидации: 0.5485
Pointwise NDCG@10 на тесте: 0.4929


In [12]:
# Готовим данные для LightGBM Ranking
# LightGBM требует, чтобы данные были отсортированы по qid

train_sorted = train.sort_values('qid').reset_index(drop=True)
vali_sorted = vali.sort_values('qid').reset_index(drop=True)
test_sorted = test.sort_values('qid').reset_index(drop=True)

X_train_s = train_sorted[feature_cols]
y_train_s = train_sorted['relevance']
qid_train_s = train_sorted['qid']

X_vali_s = vali_sorted[feature_cols]
y_vali_s = vali_sorted['relevance']
qid_vali_s = vali_sorted['qid']

X_test_s = test_sorted[feature_cols]
y_test_s = test_sorted['relevance']
qid_test_s = test_sorted['qid']

# Количество документов в каждом запросе (group)
train_group = train_sorted.groupby('qid').size().to_list()
vali_group = vali_sorted.groupby('qid').size().to_list()

In [13]:
model_listwise = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model_listwise.fit(
    X_train_s, y_train_s,
    group=train_group,
    eval_set=[(X_vali_s, y_vali_s)],
    eval_group=[vali_group],
    eval_metric='ndcg',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# Предсказания
vali_pred_list = model_listwise.predict(X_vali_s)
test_pred_list = model_listwise.predict(X_test_s)

# Считаем NDCG
ndcg_vali_list = calculate_ndcg(y_vali_s, vali_pred_list, qid_vali_s, k=10)
ndcg_test_list = calculate_ndcg(y_test_s, test_pred_list, qid_test_s, k=10)

print(f"Listwise (LambdaRank) NDCG@10 на валидации: {ndcg_vali_list:.4f}")
print(f"Listwise (LambdaRank) NDCG@10 на тесте: {ndcg_test_list:.4f}")

/Users/ivan/Desktop/learning-to-rank/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002339 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9233
[LightGBM] [Info] Number of data points in the train set: 9630, number of used features: 40
Listwise (LambdaRank) NDCG@10 на валидации: 0.5454
Listwise (LambdaRank) NDCG@10 на тесте: 0.4946
